# 01 · Data audit

**Purpose:** understand the data before modelling. What do we have, how complete is it, and what limitations must the write-up mention?

| Part | Data | Status |
|---|---|---|
| A | EPC register: social rented homes in England (the homes we will **score**) | ✅ |
| B | Geography: LSOA / MSOA / local authority | ✅ |
| C | English Housing Survey (the homes we will **train** on) | ⏳ awaiting UK Data Service access |

Inputs are built by the pipeline (`src/ingest/*`, `src/features/epc.py`, `src/geo/boundaries.py`). This notebook only reads them.

In [ ]:
from pathlib import Path

import duckdb
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SOCIAL = (ROOT / "data/processed/epc_social.parquet").as_posix()
GEO = ROOT / "data/processed/geo"

con = duckdb.connect()
con.sql(f"create or replace view social as select * from read_parquet('{SOCIAL}')")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
ORANGE = "#D85A30"


def q(sql: str) -> pd.DataFrame:
    return con.sql(sql).df()

## A. EPC: social rented homes

The cleaned table keeps **the latest certificate per property (UPRN)** in England, then keeps it only if that latest certificate says *rented (social)*. A home that was social but has since been sold therefore drops out.

In [ ]:
q('''
select count(*)                                            as homes,
       count(lsoa21cd)                                     as matched_to_lsoa,
       round(100.0 * count(lsoa21cd) / count(*), 2)        as pct_matched,
       count(distinct msoa21cd)                            as msoas,
       count(distinct lad24cd)                             as local_authorities,
       round(median(cert_age_years), 1)                    as median_cert_age_yrs
from social
''')

### How old are the certificates?

Each home is described by its *most recent* certificate, which may be years old. Older certificates may not reflect later retrofit work, so this is a limitation to report.

In [ ]:
ages = q("select year(lodgement_date) as year, count(*) as homes from social group by 1 order by 1")
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(ages.year, ages.homes / 1000, color=ORANGE)
ax.set(title="Most social homes have a certificate from 2021 or later", xlabel="Year of latest certificate", ylabel="Homes (thousands)")
plt.show()
share_recent = ages.loc[ages.year >= 2021, "homes"].sum() / ages.homes.sum()
print(f"{share_recent:.0%} of homes have a certificate lodged in 2021 or later")

### What are the homes like?

These are the **harmonised features**: coarse categories chosen so they can be matched to English Housing Survey variables. The parsing rules are in `src/features/epc.py` and are unit-tested in `tests/test_epc_features.py`.

In [ ]:
features = ["dwelling_type", "age_band", "wall_type", "wall_insulation", "heating",
            "roof_insulation", "floor_type", "glazing", "flat_position"]
orders = {"age_band": ["pre1919", "1919-1944", "1945-1964", "1965-1980", "1981-1990", "post1990"]}

fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ax, col in zip(axes.flat, features):
    d = q(f"select coalesce({col}, '(missing)') as v, count(*) as n from social group by 1").set_index("v")["n"]
    d = d.reindex([o for o in orders[col] if o in d.index]) if col in orders else d.sort_values()
    (d / d.sum() * 100).plot.barh(ax=ax, color=ORANGE)
    ax.set(title=col, xlabel="% of homes", ylabel="")
fig.suptitle("Social homes in England: harmonised EPC features", y=1.0)
fig.tight_layout()
plt.show()

### Energy efficiency

SAP score runs from 1 to 100+ (higher is more efficient). Scores above 100 occur when solar panels export energy. They're rare, so cap them at 100 for modelling.

In [ ]:
sap = q("select sap_score from social where sap_score is not null")
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(sap.sap_score.clip(upper=110), bins=110, color=ORANGE)
ax.set(title="Most social homes are EPC band C (SAP 69-80)", xlabel="SAP score (capped at 110 for display)", ylabel="Homes")
plt.show()
q("select epc_band, count(*) as homes, round(100.0 * count(*) / sum(count(*)) over (), 1) as pct from social group by 1 order by 1")

### Data quality checks

In [ ]:
q('''
select
  count(*) filter (where sap_score > 100)                         as sap_over_100,
  count(*) filter (where floor_area_m2 is null)                   as floor_area_missing,
  count(*) filter (where wall_insulation is null)                 as wall_insulation_unknown,
  count(*) filter (where roof_type <> 'dwelling_above' and roof_insulation is null) as roof_insulation_unknown,
  count(*) filter (where heating = 'other')                       as heating_unparsed,
  count(*) filter (where lsoa21cd is null)                        as no_lsoa
from social
''').T.rename(columns={0: "homes"})

## B. Geography

MSOA is the default small-area level (decided 2026-11-09): every MSOA in England has social homes, and very few have under 10.

In [ ]:
per_msoa = q("select msoa21cd, count(*) as homes from social where msoa21cd is not null group by 1")
msoa = gpd.read_file(GEO / "msoa_england.gpkg").merge(per_msoa, on="msoa21cd", how="left")

fig, ax = plt.subplots(figsize=(7, 8))
msoa.plot(column="homes", ax=ax, cmap="Oranges", scheme="quantiles", k=5, legend=True,
          legend_kwds={"title": "Social homes (quintiles)", "loc": "upper left"}, linewidth=0)
ax.set_axis_off()
ax.set_title("Social homes with an EPC, by MSOA")
plt.show()
print(msoa.homes.describe().round(0))

## C. English Housing Survey (to do)

When UK Data Service access is approved:
1. Download the EUL *Housing Stock Data* for **only odd or only even years**, skipping 2020 and 2021 (COVID).
2. Confirm the damp / mould / condensation variables in the data dictionary.
3. Map each EHS variable to the EPC features above (`dwelling_type`, `age_band`, `wall_type`, `wall_insulation`, `heating`, `roof_insulation`, `glazing`, `sap_score`, `floor_area_m2`, `region_code`).
4. Compare distributions for **social renters only** (EHS weighted vs EPC), since big differences mean the model may not transfer well.

## Findings so far
- **2.86m social homes** in England have an EPC; 99.98% are matched to small areas.
- Coverage is roughly 70% of the social stock. Homes with no certificate, or with tenure missing on their latest one, are absent.
- Features are well populated after parsing (under 0.1% unknown, except roof insulation where depth isn't recorded).